# Notebook 08 — DPO Pipeline (Kaggle GPU)

**Purpose**: Construct hallucination preference pairs and fine-tune Llama-3 8B via **QLoRA DPO** to penalize ungrounded scientific claims.

**IMPORTANT**: This notebook is designed to run on **Kaggle P100/T4 GPU** (16 GB VRAM).  
Run on Kaggle: upload this notebook + `preference_pairs.jsonl` as a Kaggle Dataset. Use Kaggle Secrets for API keys.

**Pipeline**:
1. Load pre-generated preference pairs from `data/preference_pairs.jsonl`
2. Load Llama-3 8B in 4-bit NF4 quantization
3. Configure QLoRA adapter (rank=64, target: q/k/v/o projections)
4. Train with `trl.DPOTrainer` (β=0.1, 3 epochs)
5. Export LoRA adapter weights → load locally for inference

**Outputs**:
- `data/preference_pairs.jsonl` — DPO training pairs
- `lora-adapter/` — LoRA weights for Llama-3 8B

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right) \right]$$

In [ ]:
# == Environment + env-aware paths + OpenRouter/HF keys ========================
import os
from pathlib import Path
import tempfile

# Must be set before torch initializes a CUDA context. Reduces allocator fragmentation
# (we hit an OOM failing to allocate just 10MB while the T4 was ~99.9% full).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IS_KAGGLE = os.path.exists("/kaggle")

# precompute_ref_log_probs=True writes a large Arrow cache file mid-run, then
# immediately re-opens it (trl's _precompute_ref_logps -> Dataset.from_file(cache_file)).
# On Kaggle that file defaulted to /tmp/hf_datasets-<random>/cache-<hash>.arrow, and
# after an hour-plus computing it, the re-open failed with FileNotFoundError -- /tmp on
# Kaggle is small/ephemeral and not a reliable place to park a multi-GB cache file that
# gets read back moments later. Redirecting HF datasets' cache dir, plus the stdlib
# tempfile default some internal paths fall back to, onto /kaggle/working -- the same
# large, session-persistent disk already used for OUT_DIR below. This must happen
# before `datasets` (or anything importing it) is first imported anywhere, so it's done
# here at the very top of the notebook's first cell.
if IS_KAGGLE:
    _CACHE_ROOT = Path("/kaggle/working/hf_cache")
    _CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault("HF_DATASETS_CACHE", str(_CACHE_ROOT))
    os.environ.setdefault("HF_HOME", "/kaggle/working/hf_home")
    os.environ.setdefault("TMPDIR", str(_CACHE_ROOT))
    os.environ.setdefault("TEMP", str(_CACHE_ROOT))
    os.environ.setdefault("TMP", str(_CACHE_ROOT))
    tempfile.tempdir = str(_CACHE_ROOT)  # stdlib tempfile honors this directly too

import sys, json, re, warnings
import torch
warnings.filterwarnings("ignore")

OUT_DIR   = Path("/kaggle/working") if IS_KAGGLE else Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def _find(name):
    for root in ["/kaggle/input", "/kaggle/working", "..", "."]:
        r = Path(root)
        if r.exists():
            hits = list(r.rglob(name))
            if hits:
                return hits[0]
    return None

# Keys: Kaggle Secrets first, then env. Project standard = OpenRouter (NOT OpenAI).
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    def _secret(k):
        try:
            return _sec.get_secret(k)
        except Exception:
            return ""
except Exception:
    def _secret(k):
        return os.getenv(k, "")

OPENROUTER_API_KEY  = _secret("OPENROUTER_API_KEY") or os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_BASE     = "https://openrouter.ai/api/v1"
GEN_MODEL           = "meta-llama/llama-3.3-70b-instruct"
OPENROUTER_PROVIDER = "Novita"

# The base model for QLoRA DPO below is NousResearch/Meta-Llama-3-8B-Instruct, an
# UNGATED mirror of the identical meta-llama/Meta-Llama-3-8B-Instruct weights. We
# switched to this from the official meta-llama repo after Meta's manual gated-access
# review didn't clear (confirmed via a direct hf_hub_download bypass, not just a token
# issue). HF_TOKEN is still picked up here in case it's needed elsewhere, but it is NOT
# required for the mirror.
HF_TOKEN = _secret("HF_TOKEN") or os.getenv("HF_TOKEN", "")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Kaggle:", IS_KAGGLE, "| Device:", DEVICE,
      "| OpenRouter key:", bool(OPENROUTER_API_KEY), "| HF token:", bool(HF_TOKEN))
if IS_KAGGLE:
    print("HF_DATASETS_CACHE:", os.environ.get("HF_DATASETS_CACHE"))
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: no GPU — the DPO training cell will be skipped (run on Kaggle P100/T4).")


## 1. Graph-grounded preference pairs

The mitigation is tied to the paper's contribution (CiteCheck):
- **`rejected`** = a reference rationale containing a relational claim CiteCheck flagged
  `unsupported` (a relational hallucination — a claimed paper-paper relation with **no
  citation evidence**).
- **`chosen`** = the same rationale with the flagged relational claim(s) **removed**
  (grounded); optionally rewritten for fluency via OpenRouter.

So DPO learns to **avoid ungrounded relational claims** — directly aligning mitigation with
the graph-grounded verifier. Needs `scireqa_full_relational_v1.1.jsonl` + `citecheck_results.json`
(nb01). **No human gold required** (uses CiteCheck's automatic verdicts).

In [5]:
# == Load relational benchmark + CiteCheck verdicts ===========================
# NOTE: `assert BENCH` on a bare Path(...) is always truthy even if the file doesn't
# exist -- Path() never returns falsy. Checking .exists() explicitly, and falling back
# to the robust _find() search (matches nb02's pattern) if the fixed path isn't there.
_BENCH_FIXED = Path("/kaggle/input/datasets/joytuntonny/rationale-subset/scireqa_full_relational_v1.1.jsonl")
BENCH = _BENCH_FIXED if _BENCH_FIXED.exists() else _find("scireqa_full_relational_v1.1.jsonl")
assert BENCH and Path(BENCH).exists(), "attach the dataset containing scireqa_full_relational_v1.1.jsonl"
bench = [json.loads(l) for l in open(BENCH, encoding="utf-8") if l.strip()]

_CITE_FIXED = Path("/kaggle/input/datasets/joytuntonny/rationale-subset/citecheck_results.json")
CITE = _CITE_FIXED if _CITE_FIXED.exists() else _find("citecheck_results.json")
assert CITE and Path(CITE).exists(), "attach citecheck_results.json (from 01_citecheck) - needed for graph-grounded pairs"
cite = {}
for r in json.load(open(CITE, encoding="utf-8")):
    cite[f"{r['id']}|{r['claim_id']}"] = r["verdict"]
print(f"BENCH: {BENCH}")
print(f"CITE : {CITE}")
print(f"benchmark instances: {len(bench)} | CiteCheck verdicts: {len(cite)}")


BENCH: /kaggle/input/datasets/joytuntonny/rationale-subset/scireqa_full_relational_v1.1.jsonl
CITE : /kaggle/input/datasets/joytuntonny/rationale-subset/citecheck_results.json
benchmark instances: 10000 | CiteCheck verdicts: 11517


In [6]:
# == Pair helpers ==============================================================
import requests
RUN_LLM_REWRITE = False   # set True to fluently rewrite 'chosen' via OpenRouter (costs credit); False = deterministic (free)

def _sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(text).strip()) if s.strip()]

def rewrite_grounded(text):
    """Optionally make the pruned 'chosen' read fluently WITHOUT adding any new claim/relation."""
    if not (RUN_LLM_REWRITE and OPENROUTER_API_KEY):
        return text
    prompt = ("Rewrite the following recommendation rationale to read smoothly in 2-3 sentences, "
              "WITHOUT adding any new claims or relationships between papers:\n\n" + text)
    body = {"model": GEN_MODEL, "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 150, "temperature": 0.2,
            "provider": {"order": [OPENROUTER_PROVIDER], "allow_fallbacks": False}}
    try:
        r = requests.post(f"{OPENROUTER_BASE}/chat/completions",
                          headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"}, json=body, timeout=60)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"].strip()
    except Exception:
        return text

print("Graph-grounded pair helpers defined. RUN_LLM_REWRITE =", RUN_LLM_REWRITE)

Graph-grounded pair helpers defined. RUN_LLM_REWRITE = False


In [7]:
# == Build graph-grounded preference pairs ====================================
from tqdm import tqdm
preference_pairs = []
for inst in tqdm(bench, desc="Building graph-grounded pairs"):
    rationale = (inst.get("reference_rationale") or "").strip()
    labels = inst.get("hallucination_labels", [])
    if not rationale or not labels:
        continue
    # claim_texts CiteCheck flagged as unsupported RELATIONAL claims (graph-grounded signal)
    flagged = set()
    for lbl in labels:
        rel = lbl.get("relational")
        if rel and cite.get(f"{inst['id']}|{lbl['claim_id']}") == "unsupported":
            flagged.add((lbl.get("claim_text") or "").strip())
    if not flagged:
        continue
    kept = [s for s in _sentences(rationale) if s not in flagged]
    if not kept:                      # don't emit an empty 'chosen'
        continue
    chosen   = rewrite_grounded(" ".join(kept))
    rejected = rationale              # original, contains the ungrounded relational claim
    if chosen.strip() == rejected.strip():
        continue
    prompt = ("Recommend papers for the query and explain why, grounded in the abstracts.\n"
              f"Query: {inst.get('query','')[:200]}")
    preference_pairs.append({"instance_id": inst["id"], "prompt": prompt,
                             "chosen": chosen, "rejected": rejected,
                             "n_flagged_relational": len(flagged), "domain": inst.get("domain", "")})

print(f"Built {len(preference_pairs)} graph-grounded pairs "
      f"(each 'rejected' has >=1 CiteCheck-flagged relational hallucination)")

Building graph-grounded pairs: 100%|██████████| 10000/10000 [00:00<00:00, 42276.01it/s]

Built 6582 graph-grounded pairs (each 'rejected' has >=1 CiteCheck-flagged relational hallucination)


In [8]:
# == Filter + save =============================================================
filtered_pairs = [p for p in preference_pairs
                  if p["chosen"] != p["rejected"] and len(p["chosen"]) > 30 and len(p["rejected"]) > 30]
print(f"After filtering: {len(filtered_pairs)} pairs")

pairs_path = OUT_DIR / "preference_pairs.jsonl"
with open(pairs_path, "w", encoding="utf-8") as f:
    for p in filtered_pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print("Saved:", pairs_path)

if filtered_pairs:
    p = filtered_pairs[0]
    print("\nSample pair:")
    print("  prompt  :", p["prompt"][:90])
    print("  chosen  :", p["chosen"][:160])
    print("  rejected:", p["rejected"][:160])

After filtering: 6582 pairs
Saved: /kaggle/working/preference_pairs.jsonl

Sample pair:
  prompt  : Recommend papers for the query and explain why, grounded in the abstracts.
Query: I recent
  chosen  : The recommended papers relate to each other and the query paper "Understanding Classifier Mistakes with Generative Models" through their focus on improving the 
  rejected: The recommended papers relate to each other and the query paper "Understanding Classifier Mistakes with Generative Models" through their focus on improving the 


## 2. QLoRA DPO Training

**Requires**: Kaggle P100/T4 GPU, `transformers`, `peft`, `trl`, `bitsandbytes`

In [9]:
# ── Check + INSTALL required packages (minimal, non-destructive) ===============
# NOTE: this cell previously tried to force-pin transformers/trl/peft/accelerate to a
# "known compatible" combination from mid-2024 -- WRONG assumption. This Kaggle image's
# actual baseline is numpy 2.x / torch 2.13 / CUDA 13 (visible in the flood of
# dependency-conflict warnings that pin caused: jax, cupy, cudf, opencv, etc. all expect
# numpy>=2.0). Forcing an old numpy<2.0-era combo downgraded numpy and broke dozens of
# unrelated pre-installed packages, which is why bitsandbytes' CUDA library went missing
# too. After a factory reset, DO NOT re-pin anything -- only install what's genuinely
# missing, with no version constraints, so pip's resolver picks something compatible
# with whatever Kaggle already shipped, instead of us guessing at an older ecosystem.
import subprocess, sys

required = ["transformers", "peft", "trl", "bitsandbytes", "datasets"]
missing = []
for pkg in required:
    try:
        mod = __import__(pkg)
        print(f"  already present: {pkg} == {getattr(mod, '__version__', '?')}")
    except ImportError:
        missing.append(pkg)
        print(f"  missing: {pkg}")

if missing:
    print(f"\nInstalling ONLY what's missing, no version pins: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)
else:
    print("\nNothing missing -- not touching the environment.")

# Report what's actually on the box now, for diagnosing any future incompatibility
# against THIS environment's real versions rather than assumptions.
for pkg in ["numpy", "pandas", "torch", "transformers", "peft", "trl", "accelerate", "bitsandbytes"]:
    try:
        mod = __import__(pkg)
        print(f"  {pkg:14s}: {getattr(mod, '__version__', '?')}")
    except ImportError:
        print(f"  {pkg:14s}: NOT INSTALLED")

# Exercise the EXACT imports the training cell below uses, so any incompatibility shows
# up here with a clear message instead of after the 8B model has started loading.
print()
still_broken = []
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    print("  OK: transformers (AutoTokenizer/AutoModelForCausalLM/BitsAndBytesConfig)")
except Exception as e:
    still_broken.append("transformers"); print(f"  FAILED: transformers: {type(e).__name__}: {e}")

try:
    from peft import LoraConfig, get_peft_model, TaskType
    print("  OK: peft (LoraConfig/get_peft_model/TaskType)")
except Exception as e:
    still_broken.append("peft"); print(f"  FAILED: peft: {type(e).__name__}: {e}")

try:
    from trl import DPOTrainer, DPOConfig
    print("  OK: trl (DPOTrainer/DPOConfig)")
except Exception as e:
    still_broken.append("trl"); print(f"  FAILED: trl: {type(e).__name__}: {e}")

try:
    import bitsandbytes
    print("  OK: bitsandbytes")
except Exception as e:
    still_broken.append("bitsandbytes"); print(f"  FAILED: bitsandbytes: {type(e).__name__}: {e}")

try:
    from datasets import Dataset
    print("  OK: datasets (Dataset)")
except Exception as e:
    still_broken.append("datasets"); print(f"  FAILED: datasets: {type(e).__name__}: {e}")

if still_broken:
    print(f"\n{still_broken} won't import cleanly even on a freshly-reset environment "
          f"with only the missing packages installed. If this happens, report the exact "
          f"error -- do NOT force-reinstall or pin versions as a first response, since "
          f"that's what caused the cascading breakage last time.")
else:
    print("\nAll imports the training cell needs are working.")


  already present: transformers == 5.0.0
  already present: peft == 0.19.1
  missing: trl
  missing: bitsandbytes
  already present: datasets == 5.0.0

Installing ONLY what's missing, no version pins: ['trl', 'bitsandbytes']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00
  numpy         : 2.0.2
  pandas        : 2.3.3
  torch         : 2.10.0+cu128
  transformers  : 5.0.0
  peft          : 0.19.1
  trl           : 1.8.0
  accelerate    : 1.13.0
  bitsandbytes  : 0.49.2

  OK: transformers (AutoTokenizer/AutoModelForCausalLM/BitsAndBytesConfig)
  OK: peft (LoraConfig/get_peft_model/TaskType)
  OK: trl (DPOTrainer/DPOConfig)
  OK: bitsandbytes
  OK: datasets (Dataset)

All imports the training cell needs are working.


In [ ]:
# ── DPO training (runs on Kaggle GPU; CPU path is a stub) ────────────────────
LORA_DIR = OUT_DIR / "lora-adapter"

if DEVICE == "cuda" and len(filtered_pairs) >= 10:
    import inspect
    from collections import Counter
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, TaskType
    from trl import DPOTrainer, DPOConfig
    from datasets import Dataset

    # Ungated mirror of the identical meta-llama/Meta-Llama-3-8B-Instruct weights (see
    # cell 1) -- no HF_TOKEN / gated-access approval required.
    MODEL_ID = "NousResearch/Meta-Llama-3-8B-Instruct"
    print(f"Loading base model: {MODEL_ID}")

    # torch.cuda.is_bf16_supported() returned True on a T4 in an earlier run (it only
    # checks that bf16 ops won't crash), but a T4 is Turing (compute capability 7.5) and
    # has NO bf16 tensor cores -- real hardware bf16 acceleration needs Ampere+ (compute
    # capability >= 8.0: A100, RTX 30xx and newer). Checking compute capability directly
    # so bnb_4bit_compute_dtype / torch_dtype are correct regardless of the training
    # precision decision made below.
    major, _minor = torch.cuda.get_device_capability()
    HW_BF16 = major >= 8
    compute_dtype = torch.bfloat16 if HW_BF16 else torch.float16
    print(f"GPU compute capability: {major}.{_minor} -> bf16 tensor cores: {HW_BF16} "
          f"-> using compute dtype {compute_dtype}")

    # NOTE: we tried fp16 mixed precision (bf16=False, fp16=True) here across multiple
    # attempts, including a genuinely fresh notebook (ruling out stale session state),
    # and DPOTrainer's reference-log-prob computation kept producing bf16 tensors that
    # PyTorch's fp16-only GradScaler can't unscale ("not implemented for BFloat16"),
    # regardless of the model's own dtype being forced to fp16. Rather than keep
    # chasing which internal trl/transformers code path independently decides bf16 is
    # "supported" (probably the same loose is_bf16_supported() check that misled us on
    # the compute-dtype side earlier), we disable mixed precision entirely. Without
    # fp16=True, the Trainer never creates a GradScaler, so this error class becomes
    # structurally impossible. The LoRA adapter is tiny (~54M params, <1% of the 8B
    # model) so running its updates in fp32 costs a trivial amount of extra memory; the
    # frozen base model stays 4-bit quantized regardless, since that's governed
    # separately by bnb_4bit_compute_dtype above, not by this flag.
    USE_BF16 = False

    # 4-bit NF4 quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=compute_dtype,
    )

    # LoRA configuration
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=64,
        lora_alpha=128,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    # Required alongside gradient checkpointing on a quantized+LoRA model: the base
    # model's input embeddings are frozen, so without this the checkpointed activations
    # have no requires_grad=True path back to the trainable LoRA weights and gradients
    # silently don't flow (a well-known QLoRA + gradient-checkpointing footgun).
    model.enable_input_require_grads()
    model.print_trainable_parameters()

    # Manuscript design: "training on the arXiv and ACL domains and evaluating
    # zero-shot on the held-out PubMed domain." filtered_pairs (built from the full
    # benchmark) includes all three domains -- that's correct as the full construction
    # artifact (matches the paper's "N=6,582 pairs... across the three domains"
    # description of the CONSTRUCTION), but PubMed pairs must NOT be used for actual
    # training, or the zero-shot claim is invalidated by data leakage. Filter here,
    # right at the point of building the training set, leaving filtered_pairs itself
    # (used for the saved artifact and the statistics plot above) untouched.
    train_pairs = [p for p in filtered_pairs if p.get("domain") != "pubmed"]
    _excluded = len(filtered_pairs) - len(train_pairs)
    print(f"Training pairs (arXiv + ACL only, PubMed held out for zero-shot eval): "
          f"{len(train_pairs)} (excluded {_excluded} PubMed pairs)")
    print(f"  by domain: {dict(Counter(p['domain'] for p in train_pairs))}")

    # Build HF Dataset
    train_data = Dataset.from_list([
        {"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]}
        for p in train_pairs
    ])

    # DPOConfig's accepted fields have drifted across trl versions (max_prompt_length was
    # dropped in this installed version, after max_length/max_prompt_length/tokenizer=
    # already drifted earlier). Rather than special-case each renamed/removed field as it
    # breaks, build the FULL desired kwargs and filter to whatever this installed
    # DPOConfig.__init__ actually accepts, so a future rename degrades gracefully (falls
    # back to trl's own default for that field) instead of crashing the whole run.
    #
    # OOM fix (separate issue, already applied): per_device_train_batch_size 2->1,
    # gradient_accumulation_steps 4->8 (keeps the same effective batch size of 8 and the
    # same total optimizer-step count), plus gradient checkpointing.
    #
    # precompute_ref_log_probs: LEFT DISABLED (False) on purpose. Enabling it was tried
    # twice on this exact trl 1.8.0 build: both times the reference-log-probs pass ran
    # to completion (>1hr of GPU time each time, once under /tmp, once redirected to
    # /kaggle/working/hf_cache -- ruling out disk location as the cause), and both times
    # trl's own _precompute_ref_logps then immediately failed trying to re-open the
    # cache Arrow file it had just written (FileNotFoundError inside
    # concatenate_datasets([...,  Dataset.from_file(cache_file)], axis=1)). That's a bug
    # in this trl build's caching path for a single-process (non `accelerate launch`)
    # run, not something fixable from notebook code. Reference log-probs are instead
    # computed inline every step (the trl default) -- slower per step, but this code
    # path has actually completed successfully before (the very first working run,
    # pre-precompute), so it's the reliable choice given repeated GPU-hour losses on the
    # alternative. If a future trl release fixes the caching bug, flip this back to True.
    desired_dpo_kwargs = dict(
        output_dir=str(LORA_DIR),
        beta=0.1,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        learning_rate=5e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        max_length=512,
        max_prompt_length=256,
        logging_steps=10,
        save_steps=50,
        bf16=False,
        fp16=False,
        precompute_ref_log_probs=False,
        report_to="none",
    )
    _accepted = set(inspect.signature(DPOConfig.__init__).parameters)
    dropped = {k: v for k, v in desired_dpo_kwargs.items() if k not in _accepted}
    dpo_kwargs = {k: v for k, v in desired_dpo_kwargs.items() if k in _accepted}
    if dropped:
        print(f"NOTE: this trl version's DPOConfig doesn't accept {list(dropped)} -- "
              f"using its defaults for those instead. ({dropped})")
    dpo_config = DPOConfig(**dpo_kwargs)

    # trl renamed DPOTrainer's `tokenizer=` kwarg to `processing_class=` around 0.12;
    # try the older-API name first and fall back so this works with whatever trl
    # version pip resolved to in the install cell above.
    try:
        trainer = DPOTrainer(model=model, args=dpo_config, train_dataset=train_data, tokenizer=tokenizer)
    except TypeError:
        trainer = DPOTrainer(model=model, args=dpo_config, train_dataset=train_data, processing_class=tokenizer)

    # Confirms no mixed precision is active -- should print "no" (accelerate's
    # mixed_precision value when disabled).
    try:
        print(f"Accelerator mixed_precision in effect: {trainer.accelerator.mixed_precision}")
    except Exception as e:
        print(f"(could not introspect accelerator mixed_precision: {e})")

    _eff_batch = dpo_kwargs.get("per_device_train_batch_size", 1) * dpo_kwargs.get("gradient_accumulation_steps", 1)
    _steps = (len(train_pairs) * dpo_kwargs.get("num_train_epochs", 1)) // max(_eff_batch, 1)
    print(f"~{_steps} optimizer steps expected (no ref-log-prob precompute -- reference "
          f"model recomputed every step, so this will run slower than the it/s you may "
          f"have seen with precompute enabled).")

    print("Starting DPO training...")
    trainer.train()

    # Save LoRA adapter
    LORA_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(LORA_DIR))
    tokenizer.save_pretrained(str(LORA_DIR))
    print(f"LoRA adapter saved: {LORA_DIR}")

else:
    print("[CPU / insufficient pairs] Skipping DPO training.")
    print("  → Run this cell on Kaggle GPU with full preference_pairs.jsonl.")
    print("  → After training, download lora-adapter/ directory.")


## 3. Local Inference with LoRA Adapter

After exporting `lora-adapter/` from Kaggle, load it locally for inference.

In [ ]:
def load_dpo_model(adapter_path: str = "../lora-adapter"):
    """Load the DPO-fine-tuned Llama-3 8B with LoRA adapter for local inference."""
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel

    # Ungated mirror of meta-llama/Meta-Llama-3-8B-Instruct -- see cell 1's note.
    MODEL_ID = "NousResearch/Meta-Llama-3-8B-Instruct"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()
    return tokenizer, model


def generate_dpo_rationale(tokenizer, model, prompt: str, max_new_tokens: int = 150) -> str:
    """Generate a rationale with the DPO-aligned model."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Usage (after downloading lora-adapter/ from Kaggle):
# tokenizer, dpo_model = load_dpo_model("../lora-adapter")
# rationale = generate_dpo_rationale(tokenizer, dpo_model, prompt)

print("DPO inference helpers defined.")
print("Load model after downloading lora-adapter/ from Kaggle.")


## 4. Preference Pair Statistics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

if filtered_pairs:
    domain_counts = Counter(p["domain"] for p in filtered_pairs)
    chosen_lens   = [len(p["chosen"].split()) for p in filtered_pairs]
    rejected_lens = [len(p["rejected"].split()) for p in filtered_pairs]
    nflag         = [p.get("n_flagged_relational", 0) for p in filtered_pairs]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(domain_counts.keys(), domain_counts.values(), color="steelblue")
    axes[0].set_title("Graph-grounded pairs by domain"); axes[0].set_ylabel("Count")
    axes[1].hist(chosen_lens, bins=20, alpha=0.7, label="chosen (grounded)", color="green")
    axes[1].hist(rejected_lens, bins=20, alpha=0.7, label="rejected (w/ ungrounded relation)", color="red")
    axes[1].set_title("Rationale length"); axes[1].set_xlabel("word count"); axes[1].legend()
    plt.tight_layout(); plt.savefig(OUT_DIR / "fig_preference_pairs.png"); plt.show()

    print(f"Total pairs: {len(filtered_pairs)} | domains: {dict(domain_counts)}")
    print(f"Avg chosen {np.mean(chosen_lens):.1f}w | rejected {np.mean(rejected_lens):.1f}w "
          f"| flagged relational/pair {np.mean(nflag):.2f}")

## 5. Summary

| Step | Description | Location |
|---|---|---|
| Pair generation | **Graph-grounded**: `rejected` = rationale with a CiteCheck-flagged relational hallucination; `chosen` = same rationale with the flagged relation removed | Local / Kaggle |
| Base model | Llama-3 8B Instruct (4-bit NF4), via the `NousResearch/Meta-Llama-3-8B-Instruct` ungated mirror | Kaggle GPU |
| LoRA config | rank=64, α=128, targets=q/k/v/o | Kaggle GPU |
| DPO training | β=0.1, 3 epochs, batch=2×4 grad accum | Kaggle GPU |
| Export | `lora-adapter/` → download | Kaggle → Local |

DPO penalizes **ungrounded relational claims**, tying the mitigation to CiteCheck (no human gold needed).
Note for the manuscript's reproducibility/methods section: the base model is loaded from
`NousResearch/Meta-Llama-3-8B-Instruct`, a widely-used unofficial mirror of the identical
`meta-llama/Meta-Llama-3-8B-Instruct` weights, used here because Meta's manual gated-access
review did not clear during development.

**Next**: Notebook 09 — Evaluation & Ablation (redesign pending human gold).